# Stage 10 — baseline model training and evaluation

Train Logistic Regression, Random Forest, and HistGradientBoosting on (1) static SIM profiles and (2) causal SIM-hour history features. All partitions are grouped by SIM. Hyperparameters are fixed in advance; the decision threshold is selected on validation data by F1 and evaluated once on the held-out test data. Confusion matrices, detailed metrics, classification reports, and test predictions are saved for every model.

Rerun Stage 9 first to ensure the history table does not contain full-day static aggregates.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,precision_score,recall_score,f1_score,average_precision_score,roc_auc_score,confusion_matrix,classification_report)

SEED=42; BASE=Path('../outputs/model_ready/all_naive_12pct_200sims'); OUT=Path('../outputs/models/baseline_comparison'); OUT.mkdir(parents=True,exist_ok=True)
static=pd.read_csv(BASE/'static_sim_features.csv',dtype={'sim':'string'})
history=pd.read_csv(BASE/'history_aware_sim_hour_features.csv',dtype={'sim':'string'},parse_dates=['hour_ts'])
print('static rows:',len(static),'history rows:',len(history))

In [ ]:
def make_models():
    return {
      'Logistic Regression':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=3000,random_state=SEED)),
      'Random Forest':make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(n_estimators=500,min_samples_leaf=2,class_weight='balanced_subsample',n_jobs=-1,random_state=SEED)),
      'HistGradientBoosting':make_pipeline(SimpleImputer(strategy='median'),HistGradientBoostingClassifier(max_iter=250,learning_rate=.06,max_leaf_nodes=15,l2_regularization=1.0,random_state=SEED))
    }

def grouped_partitions(X,y,groups):
    # 60% train, 20% validation, 20% test, by SIM group.
    first=GroupShuffleSplit(n_splits=1,test_size=.40,random_state=SEED)
    train_idx,remain_idx=next(first.split(X,y,groups))
    second=GroupShuffleSplit(n_splits=1,test_size=.50,random_state=SEED+1)
    val_rel,test_rel=next(second.split(X.iloc[remain_idx],y.iloc[remain_idx],groups.iloc[remain_idx]))
    return train_idx,remain_idx[val_rel],remain_idx[test_rel]

def feature_matrix(frame,target,group):
    excluded={target,group,'hour_ts','known_fraud','static_fraud','record_id','event_key','src_number','dst_number','src_imei','dst_imei'}
    cols=[c for c in frame.select_dtypes(include='number').columns if c not in excluded]
    if not cols: raise ValueError('No numeric predictors remain after identifier/target exclusions.')
    X=frame[cols].replace([np.inf,-np.inf],np.nan)
    y=frame[target].astype(int); groups=frame[group].astype(str)
    if y.nunique()!=2: raise ValueError(f'Expected both classes in {target}: {y.value_counts().to_dict()}')
    return X,y,groups,cols

In [ ]:
def select_threshold(y_true,probabilities):
    thresholds=np.linspace(.01,.99,99)
    scores=[f1_score(y_true,(probabilities>=t).astype(int),zero_division=0) for t in thresholds]
    best=int(np.argmax(scores))
    return float(thresholds[best]),float(scores[best])

def run_experiment(frame,dataset,target,group):
    X,y,groups,columns=feature_matrix(frame,target,group)
    train_idx,val_idx,test_idx=grouped_partitions(X,y,groups)
    groupsets=[set(groups.iloc[ix]) for ix in (train_idx,val_idx,test_idx)]
    assert not(groupsets[0]&groupsets[1] or groupsets[0]&groupsets[2] or groupsets[1]&groupsets[2]),'SIM leakage across partitions'
    print(dataset,'predictors:',len(columns),'SIM counts:',[len(s) for s in groupsets],'positive SIM counts:',[int(y.iloc[ix].groupby(groups.iloc[ix]).max().sum()) for ix in (train_idx,val_idx,test_idx)])
    output=[]; matrices=[]; reports=[]; predictions=[]
    for model_name,model in make_models().items():
        # Apply balanced sample weights to HistGradientBoosting; other models use native class weights.
        if model_name=='HistGradientBoosting':
            weights=compute_sample_weight(class_weight='balanced',y=y.iloc[train_idx])
            model.fit(X.iloc[train_idx],y.iloc[train_idx],histgradientboostingclassifier__sample_weight=weights)
        else:
            model.fit(X.iloc[train_idx],y.iloc[train_idx])
        val_prob=model.predict_proba(X.iloc[val_idx])[:,1]
        threshold,val_f1=select_threshold(y.iloc[val_idx],val_prob)
        test_prob=model.predict_proba(X.iloc[test_idx])[:,1]
        test_pred=(test_prob>=threshold).astype(int)
        yt=y.iloc[test_idx].to_numpy()
        cm=confusion_matrix(yt,test_pred,labels=[0,1])
        row={'dataset':dataset,'model':model_name,'threshold':threshold,'validation_f1_at_selected_threshold':val_f1,'test_accuracy':accuracy_score(yt,test_pred),'test_balanced_accuracy':balanced_accuracy_score(yt,test_pred),'test_precision':precision_score(yt,test_pred,zero_division=0),'test_recall':recall_score(yt,test_pred,zero_division=0),'test_f1':f1_score(yt,test_pred,zero_division=0),'test_pr_auc':average_precision_score(yt,test_prob),'test_roc_auc':roc_auc_score(yt,test_prob),'tn':cm[0,0],'fp':cm[0,1],'fn':cm[1,0],'tp':cm[1,1],'test_rows':len(test_idx),'test_sims':groups.iloc[test_idx].nunique(),'train_sims':groups.iloc[train_idx].nunique(),'validation_sims':groups.iloc[val_idx].nunique()}
        output.append(row)
        matrices.append({'dataset':dataset,'model':model_name,'matrix':cm})
        rep=classification_report(yt,test_pred,labels=[0,1],target_names=['legitimate','fraudulent'],output_dict=True,zero_division=0)
        for label,metrics in rep.items():
            if isinstance(metrics,dict): reports.append({'dataset':dataset,'model':model_name,'class':label,**metrics})
        ids=frame.iloc[test_idx][group].astype(str).to_numpy()
        ts=frame.iloc[test_idx]['hour_ts'].astype(str).to_numpy() if 'hour_ts' in frame else np.repeat('',len(test_idx))
        predictions.extend({'dataset':dataset,'model':model_name,'sim':ids[i],'hour':ts[i],'actual':int(yt[i]),'probability':float(test_prob[i]),'threshold':threshold,'predicted':int(test_pred[i])} for i in range(len(test_idx)))
    return output,matrices,reports,predictions,columns

In [ ]:
all_metrics=[]; all_matrices=[]; all_reports=[]; all_predictions=[]
for frame,dataset,target,group in [(static,'static_SIM','static_fraud','sim'),(history,'history_SIM_hour','fraud','sim')]:
    metrics,matrices,reports,predictions,columns=run_experiment(frame,dataset,target,group)
    all_metrics.extend(metrics); all_matrices.extend(matrices); all_reports.extend(reports); all_predictions.extend(predictions)

metrics_df=pd.DataFrame(all_metrics).sort_values(['dataset','test_pr_auc'],ascending=[True,False])
metrics_df.to_csv(OUT/'baseline_metrics.csv',index=False)
pd.DataFrame(all_reports).to_csv(OUT/'classification_reports.csv',index=False)
pd.DataFrame(all_predictions).to_csv(OUT/'test_predictions.csv',index=False)
metrics_df

In [ ]:
# Save each confusion matrix as a CSV and a labeled plot.
for item in all_matrices:
    cm=item['matrix']; slug=item['dataset']+'_'+item['model'].lower().replace(' ','_')
    pd.DataFrame(cm,index=['actual_legitimate','actual_fraudulent'],columns=['predicted_legitimate','predicted_fraudulent']).to_csv(OUT/f'confusion_matrix_{slug}.csv')
    fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(cm,cmap='Blues')
    ax.set_xticks([0,1],['Legitimate','Fraudulent']); ax.set_yticks([0,1],['Legitimate','Fraudulent'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(item['dataset']+' - '+item['model'])
    for (i,j),value in np.ndenumerate(cm): ax.text(j,i,str(value),ha='center',va='center')
    fig.colorbar(im,ax=ax); fig.tight_layout(); fig.savefig(OUT/f'confusion_matrix_{slug}.png',dpi=160); plt.show()

# Model comparison chart using held-out test PR-AUC.
fig,ax=plt.subplots(figsize=(9,5)); metrics_df.pivot(index='model',columns='dataset',values='test_pr_auc').plot(kind='bar',ax=ax)
ax.set_ylabel('Test PR-AUC'); ax.set_xlabel('Baseline model'); ax.set_title('Held-out baseline comparison'); fig.tight_layout(); fig.savefig(OUT/'baseline_test_pr_auc.png',dpi=160); plt.show()
print('Full results saved under:',OUT)

## Outputs and interpretation

`baseline_metrics.csv` contains thresholded test metrics, ranking metrics, confusion counts, and split sizes. `classification_reports.csv` contains per-class precision, recall, F1, and support. Each model and dataset has a labeled confusion matrix in CSV and PNG form. `test_predictions.csv` preserves held-out probabilities and decisions for follow-up analysis. The validation split selects the F1 threshold; test results are not used to tune it. Static results classify full-day profiles. Hourly results classify SIM-hour rows and must be summarized at SIM level for early-detection claims.